# Jev standalone — Fake.br

Execução comparativa independente conforme o plano Jev e protocolo comum. O estado enviado ao Jev contém somente texto_trunc (máximo de 200 palavras); labels, IDs, grupos, pares, origem e rótulos de fact-check permanecem no lado da avaliação.

**Gate desta execução:** skipped_no_free_credit. Em 2026-09-24 não há confirmação da equipe sobre créditos promocionais suficientes, limite de gasto pago igual a zero ou recargas automáticas desativadas, nem evidência do painel da conta. Nenhuma chamada Jev será feita. A amostra e a estimativa são congeladas antes de qualquer resposta. Para uma execução autorizada, preencha as atestações na célula de configuração após a verificação pela equipe.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, math, os, time, urllib.request, zipfile
import numpy as np
import pandas as pd
import sklearn, sys
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / ".git").exists():
    ROOT = ROOT.parent
if not (ROOT / ".git").exists():
    raise RuntimeError("Execute dentro do repositório olimpo-fake-news-ai.")
RUN_ID = "20260924T164252Z"
RUN_STARTED_UTC = datetime.now(timezone.utc).isoformat()
RUN_DIR = ROOT / "machine-learning/outputs/model-comparison" / f"jev-standalone-{RUN_ID}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
SRC = ROOT / "machine-learning/outputs/model-comparison/dbscan-20260924T141332Z"
SRC_MANIFEST, SRC_PRED, SRC_METRICS = SRC/"run_manifest.json", SRC/"predictions.csv", SRC/"metrics.csv"
ARCHIVE = ROOT / "machine-learning/unsupervised-learning/data/Fake.br-Corpus-780f5516c4ae070761632d98ac3368f3ded09d35.zip"
SEED, API_BASE, API_VERSION, MODEL_ALIAS = 42, "https://api.typesafe.ai", "0.2.0", "jev-latest"
INPUT_PRICE, OUTPUT_PRICE, CALL_CAP, TIMEOUT = 0.042, 0.0, 2880, 120
# Atestações exigem confirmação explícita da equipe; defaults falham fechados.
TEAM_CONFIRMED_PROMOTIONAL_BUDGET_USD = None
TEAM_CONFIRMS_BUDGET_COVERS_FROZEN_SAMPLE = False
TEAM_CONFIRMS_PAID_SPEND_LIMIT_IS_ZERO = False
TEAM_CONFIRMS_AUTOMATIC_REFILLS_DISABLED = False
TEAM_CONFIRMATION_REFERENCE = None
ACCOUNT_SETTINGS_CHECKED_AT_UTC = None
PRICE_SOURCES = {"checked_on":"2026-09-24","api":"https://api.typesafe.ai/docs",
 "openapi":"https://api.typesafe.ai/openapi.json","pricing":"https://typesafe.ai/",
 "pricing_announcement":"https://typesafe.ai/blog/introducing-system-one-models-and-jev",
 "credit_terms":"https://typesafe.ai/legal/mca","credit_terms_updated_on":"2026-09-19"}

manifest_src = json.loads(SRC_MANIFEST.read_text(encoding="utf-8"))
pred_src = pd.read_csv(SRC_PRED)
corpus_sha = hashlib.sha256(ARCHIVE.read_bytes()).hexdigest()
assert manifest_src["corpus"]["revision"] == "780f5516c4ae070761632d98ac3368f3ded09d35"
assert corpus_sha == manifest_src["corpus"]["archive_sha256"] == manifest_src["corpus"]["expected_sha256"]
assert manifest_src["corpus"]["labels"] == {"0":"True","1":"Fake"}
group_rows = pred_src[["record_id","group_id"]].dropna().drop_duplicates()
assert group_rows.groupby("record_id")["group_id"].nunique().max() == 1
group = dict(zip(group_rows["record_id"],group_rows["group_id"]))
parts = {}
for split in ("train","validation","test"):
    p = manifest_src["partitions"]["canonical_random_group"][split]
    parts[split] = list(p["record_ids"])
    assert set(p["group_ids"]) == {group[r] for r in parts[split]}
ids = {s:set(v) for s,v in parts.items()}
groups = {s:{group[r] for r in rs} for s,rs in ids.items()}
for a,b in (("train","validation"),("train","test"),("validation","test")):
    assert ids[a].isdisjoint(ids[b]) and groups[a].isdisjoint(groups[b])
for s in parts:
    assert sum(r.startswith("fake/") for r in parts[s]) == sum(r.startswith("true/") for r in parts[s])

prefix = "Fake.br-Corpus-780f5516c4ae070761632d98ac3368f3ded09d35/full_texts"
candidate = {"validation":parts["validation"],"test":parts["test"]}
with zipfile.ZipFile(ARCHIVE) as zf:
    archive_names = set(zf.namelist())
    def trunc(rid):
        label_dir, num = rid.split("/",1)
        member = f"{prefix}/{label_dir}/{num}.txt"
        if member not in archive_names: raise FileNotFoundError(member)
        return " ".join(zf.read(member).decode("utf-8").split()[:200])
    text = {r:trunc(r) for rs in candidate.values() for r in rs}
assert all(len(t.split())<=200 for t in text.values())

qname = "jev_assessment"
question = {"type":"choice","instructions":(
 "Classifique a alegação factual central, não o estilo. Escolha fake somente se parecer falsa ou contrariar "
 "conhecimento confiável e estabelecido disponível a você. Sensacionalismo, viés, erros ou falta de citação, "
 "isoladamente, não provam falsidade. Escolha real somente se houver suporte confiável e nenhuma incerteza "
 "material; plausibilidade ou ausência de contradição não bastam. Se o texto e seu conhecimento não trouxerem "
 "evidência confiável suficiente, houver evidência mista, ou contexto/fontes atuais forem necessários, escolha "
 "needs_verification. Não alegue consulta a fontes externas nem trate a resposta como fact-check comprovado."),
 "criteria":{"fake":"A alegação central é falsa ou contradita por conhecimento confiável estabelecido.",
 "real":"A alegação central tem apoio confiável e não há incerteza material.",
 "needs_verification":"Evidência insuficiente para decidir; é preciso verificar."}}
options = ["fake","real","needs_verification"]
def payload(rid, model=MODEL_ALIAS):
    return {"model":model,"state":text[rid],"questions":{qname:question}}
def upper_tokens(rid):
    return len(json.dumps(payload(rid),ensure_ascii=False,separators=(",",":")).encode("utf-8"))
def cost(rid): return upper_tokens(rid)*INPUT_PRICE/1_000_000

full_cost = sum(cost(r) for rs in candidate.values() for r in rs)
if TEAM_CONFIRMED_PROMOTIONAL_BUDGET_USD is None:
    sample = {s:list(v) for s,v in candidate.items()}
    sample_rule = "all eligible validation/test IDs; credit budget unverified"
else:
    budget = float(TEAM_CONFIRMED_PROMOTIONAL_BUDGET_USD)
    if not math.isfinite(budget) or budget<0: raise ValueError("Invalid verified promotional budget.")
    rng=np.random.default_rng(SEED); order={}; cumulative={}
    for s in ("validation","test"):
        for cls in ("true/","fake/"):
            pool=[r for r in parts[s] if r.startswith(cls)]
            order[s,cls]=list(np.asarray(pool,dtype=object)[rng.permutation(len(pool))])
            cumulative[s,cls]=np.cumsum([cost(r) for r in order[s,cls]])
    cap=min(len(v) for v in order.values()); n_each=0
    for n in range(cap+1):
        needed=sum(float(cumulative[k][n-1]) if n else 0.0 for k in cumulative)
        if needed<=budget: n_each=n
        else: break
    sample={s:order[s,"true/"][:n_each]+order[s,"fake/"][:n_each] for s in ("validation","test")}
    sample_rule=f"seed 42; {n_each} per class per partition"
for s in sample:
    assert sum(r.startswith("true/") for r in sample[s])==sum(r.startswith("fake/") for r in sample[s])
chosen=sample["validation"]+sample["test"]
assert len(chosen)==len(set(chosen)) and {group[r] for r in sample["validation"]}.isdisjoint({group[r] for r in sample["test"]}) and len(chosen)<=CALL_CAP
estimate=sum(cost(r) for r in chosen); token_bound=sum(upper_tokens(r) for r in chosen)

def evalbin(y,pred,score=None,prob=False):
    y,pred=np.asarray(y,int),np.asarray(pred,int)
    tp=int(((y==1)&(pred==1)).sum()); fp=int(((y==0)&(pred==1)).sum())
    tn=int(((y==0)&(pred==0)).sum()); fn=int(((y==1)&(pred==0)).sum())
    p1=tp/(tp+fp) if tp+fp else 0.; r1=tp/(tp+fn) if tp+fn else 0.
    p0=tn/(tn+fn) if tn+fn else 0.; r0=tn/(tn+fp) if tn+fp else 0.
    f=lambda p,r:2*p*r/(p+r) if p+r else 0.
    out={"n_samples":len(y),"n_true":int((y==0).sum()),"n_fake":int((y==1).sum()),
      "macro_f1":(f(p0,r0)+f(p1,r1))/2,"balanced_accuracy":(r0+r1)/2,
      "precision_fake":p1,"recall_fake":r1,"f1_fake":f(p1,r1),
      "fpr_true":fp/(fp+tn) if fp+tn else np.nan,"accuracy":(tp+tn)/len(y) if len(y) else np.nan,
      "coverage":1.,"abstention_rate":0.,"selective_accuracy":(tp+tn)/len(y) if len(y) else np.nan,
      "roc_auc":np.nan,"average_precision":np.nan,"brier_score_fake":np.nan}
    if score is not None:
        score=np.asarray(score,float)
        if np.all(np.isfinite(score)) and len(np.unique(score))>1:
            out["roc_auc"]=float(roc_auc_score(y,score)); out["average_precision"]=float(average_precision_score(y,score))
        if prob and np.all(np.isfinite(score)): out["brier_score_fake"]=float(brier_score_loss(y,score))
    return out

pred_src=pd.read_csv(SRC_PRED); source_metrics=pd.read_csv(SRC_METRICS)
specs=[("LogisticRegression","Logistic Regression"),("LinearSVC","LinearSVC"),("RandomForest","Random Forest"),
 ("IsolationForest","Isolation Forest"),("LOF","LOF"),("OneClassSVM","One-Class SVM"),
 ("PULearningSpyRF","PU Learning"),("KMeansNovelty","K-means novidade"),("DBSCANStyleNovelty","DBSCAN novidade")]
def bval(x):
    if pd.isna(x): return None
    if isinstance(x,(bool,np.bool_)): return bool(x)
    return str(x).strip().lower() in {"true","1","yes"}
basepred=[]; basemetrics=[]
for src,name in specs:
    q=pred_src[(pred_src.protocol=="canonical_random_group")&(pred_src.partition=="test")&(pred_src.method==src)].copy()
    assert set(q.record_id)==ids["test"],f"Test IDs differ for {src}"
    q=q[q.record_id.isin(set(sample["test"]))].copy()
    if q.empty:
        empty={k:np.nan for k in ("macro_f1","balanced_accuracy","precision_fake","recall_fake","f1_fake","fpr_true","accuracy","coverage","abstention_rate","selective_accuracy","roc_auc","average_precision","brier_score_fake")}
        basemetrics.append({"method":name,"protocol":"canonical_random_group","partition":"test","status":"not_evaluated_no_test_sample",
            "n_samples":0,"n_true":0,"n_fake":0,**empty,"latency_seconds":np.nan,"estimated_cost_usd":np.nan,
            "credits_consumed":"not_applicable_local_model","score_type":"unavailable",
            "metric_origin":"No Jev test sample fits the confirmed free-credit budget; local metrics not compared.",
            "source_metric_run":"dbscan-20260924T141332Z/metrics.csv"})
        continue
    q["label_for_evaluation_only"]=q.record_id.map(lambda r:int(r.startswith("fake/")))
    dcol="decision_fake" if q.decision_fake.notna().any() else "decision_q95"
    dec=q[dcol].map(bval); assert dec.notna().all(),f"Missing predictions for {src}"
    y=q.label_for_evaluation_only.to_numpy(int); pred=dec.to_numpy(int)
    score=pd.to_numeric(q.score,errors="coerce").to_numpy(float)
    stype=str(q.score_type.dropna().iloc[0]); prob=stype=="positive_class_probability"
    mm=evalbin(y,pred,score,prob)
    old=source_metrics[(source_metrics.protocol=="canonical_random_group")&(source_metrics.partition=="test")&(source_metrics.method==src)]
    if len(sample["test"])==len(parts["test"]) and not old.empty:
        for col in ("macro_f1","balanced_accuracy","accuracy"):
            if pd.notna(old.iloc[0].get(col)): assert abs(float(old.iloc[0][col])-mm[col])<1e-6,(src,col)
    basemetrics.append({"method":name,"protocol":"canonical_random_group","partition":"test","status":"executed_local_baseline",
       **mm,"latency_seconds":np.nan,"estimated_cost_usd":np.nan,"credits_consumed":"not_applicable_local_model",
       "score_type":stype,"metric_origin":"Recomputed from matching DBSCAN run predictions.csv",
       "source_metric_run":"dbscan-20260924T141332Z/metrics.csv"})
    for (_,r),pf in zip(q.iterrows(),pred):
        p=float(r.score) if prob and pd.notna(r.score) else np.nan
        basepred.append({"model":name,"partition":"test","record_id":r.record_id,
          "label_for_evaluation_only":int(r.label_for_evaluation_only),"response":"fake" if pf else "real",
          "score_fake":p,"probabilities_json":json.dumps({"fake":p,"real":1-p}) if np.isfinite(p) else None,
          "confidence":np.nan,"abstention":False,"latency_seconds":np.nan,"estimated_cost_usd":np.nan,
          "provider_reported_cost_usd":"unavailable_local_baseline","credits_consumed":"not_applicable_local_model",
          "usage_input_tokens":np.nan,"usage_output_tokens":np.nan,"model_version":"DBSCAN run local output","score":float(r.score) if pd.notna(r.score) else np.nan,
          "score_type":stype,"state_sent":None})

# Durable per-record checkpoint. in_flight/uncertain events are never retried.
checkpoint=RUN_DIR/"jev_checkpoint.jsonl"
def latest_events():
    out={}
    if checkpoint.exists():
        for line in checkpoint.read_text(encoding="utf-8").splitlines():
            if line.strip():
                e=json.loads(line); out[e["record_id"]]=e
    return out
def append_event(e):
    with checkpoint.open("a",encoding="utf-8") as f:
        f.write(json.dumps(e,ensure_ascii=False,separators=(",",":"))+"\n"); f.flush(); os.fsync(f.fileno())
def request(path,key,method="GET",body=None):
    data=None if body is None else json.dumps(body,ensure_ascii=False).encode("utf-8")
    req=urllib.request.Request(API_BASE+path,data=data,method=method,
       headers={"Authorization":f"Bearer {key}","Content-Type":"application/json"})
    with urllib.request.urlopen(req,timeout=TIMEOUT) as res: return json.loads(res.read().decode("utf-8"))
def calljev(rid,split,key,model,latest):
    if rid in latest:
        if latest[rid].get("status")=="success": return latest[rid]
        raise RuntimeError("Uncertain checkpoint; refusing duplicate request.")
    body=payload(rid,model)
    ev={"record_id":rid,"partition":split,"status":"in_flight","request_state":text[rid],
        "request_state_word_count":len(text[rid].split()),
        "request_sha256":hashlib.sha256(json.dumps(body,ensure_ascii=False,sort_keys=True).encode()).hexdigest(),
        "started_at_utc":datetime.now(timezone.utc).isoformat()}
    append_event(ev)
    try:
        start=time.perf_counter(); result=request("/v1/systemone",key,"POST",body); latency=time.perf_counter()-start
    except Exception as exc:
        bad={"record_id":rid,"partition":split,"status":"uncertain_failure","error_type":type(exc).__name__}
        append_event(bad); latest[rid]=bad
        raise RuntimeError("Outcome uncertain; stopped without retry.") from None
    answer=result.get("answers",{}).get(qname,{}); choice=answer.get("choice"); probs=answer.get("probabilities")
    if choice not in options or not isinstance(probs,dict):
        bad={"record_id":rid,"partition":split,"status":"uncertain_failure","error_type":"invalid_response_schema"}
        append_event(bad); latest[rid]=bad; raise RuntimeError("Invalid output; stopped without retry.")
    usage=result.get("usage",{}); it,ot=usage.get("input_tokens"),usage.get("output_tokens")
    ev.update({"status":"success","choice":choice,"probabilities":probs,"confidence":answer.get("confidence"),
       "model_version":result.get("model","unavailable"),"usage_input_tokens":it if it is not None else "unavailable",
       "usage_output_tokens":ot if ot is not None else "unavailable","latency_seconds":latency,
       "provider_reported_cost_usd":result.get("cost_usd","unavailable"),
       "credits_consumed":result.get("credits_consumed",result.get("credits_used","unavailable")),
       "estimated_cost_usd_from_public_rate":(it*INPUT_PRICE/1e6+ot*OUTPUT_PRICE/1e6) if isinstance(it,int) and isinstance(ot,int) else np.nan,
       "finished_at_utc":datetime.now(timezone.utc).isoformat()})
    append_event(ev); latest[rid]=ev; return ev

gate_ok=all([TEAM_CONFIRMED_PROMOTIONAL_BUDGET_USD is not None,TEAM_CONFIRMS_BUDGET_COVERS_FROZEN_SAMPLE,
 TEAM_CONFIRMS_PAID_SPEND_LIMIT_IS_ZERO,TEAM_CONFIRMS_AUTOMATIC_REFILLS_DISABLED,
 bool(TEAM_CONFIRMATION_REFERENCE),bool(ACCOUNT_SETTINGS_CHECKED_AT_UTC),
 float(TEAM_CONFIRMED_PROMOTIONAL_BUDGET_USD or 0)>=estimate,len(chosen)>0])
jev_status="skipped_no_free_credit"
jev_reason="Team promotional-credit coverage and account billing controls were not verified; no API request was made."
jev_val=[]; jev_test=[]; frozen_mapping=None; model_resolved="unavailable_no_call"
if gate_ok:
    key=os.environ.get("TYPESAFE_API_KEY")  # runtime only; never printed or serialized
    if not key: jev_reason="No team-provided TYPESAFE_API_KEY in runtime; no API request was made."
    else:
        latest=latest_events()
        if any(e.get("status") in {"in_flight","uncertain_failure"} for e in latest.values()):
            jev_status="blocked_uncertain_checkpoint"; jev_reason="Uncertain checkpoint prevents duplicate calls."
        else:
            try:
                models=request("/v1/models",key).get("models",[])
                model_resolved=next((m["name"] for m in models if m.get("name")==MODEL_ALIAS),None)
                if model_resolved is None: raise RuntimeError("Pinned model alias unavailable; no POST made.")
                for rid in sample["validation"]: jev_val.append(calljev(rid,"validation",key,model_resolved,latest))
                yv=np.array([int(e["record_id"].startswith("fake/")) for e in jev_val])
                candidates=[]
                for target,label in (("real",0),("fake",1)):
                    pv=np.array([1 if e["choice"]=="fake" else 0 if e["choice"]=="real" else label for e in jev_val])
                    candidates.append((evalbin(yv,pv)["macro_f1"],1 if target=="real" else 0,target))
                frozen_mapping=sorted(candidates,key=lambda x:(-x[0],-x[1]))[0][2]
                # Binary mapping is now frozen; only now request test responses.
                for rid in sample["test"]: jev_test.append(calljev(rid,"test",key,model_resolved,latest))
                jev_status="executed"; jev_reason=None
            except Exception as exc:
                if "uncertain" in str(exc).lower() or "checkpoint" in str(exc).lower():
                    jev_status="blocked_uncertain_checkpoint"
                jev_reason=f"Stopped ({type(exc).__name__}); no retry attempted."
else:
    missing=[]
    if TEAM_CONFIRMED_PROMOTIONAL_BUDGET_USD is None: missing.append("confirmed promotional budget")
    if not TEAM_CONFIRMS_BUDGET_COVERS_FROZEN_SAMPLE: missing.append("confirmation that free credits cover frozen sample")
    if not TEAM_CONFIRMS_PAID_SPEND_LIMIT_IS_ZERO: missing.append("zero paid-spend limit")
    if not TEAM_CONFIRMS_AUTOMATIC_REFILLS_DISABLED: missing.append("automatic refills disabled")
    if not TEAM_CONFIRMATION_REFERENCE or not ACCOUNT_SETTINGS_CHECKED_AT_UTC: missing.append("team/account verification reference and time")
    jev_reason="Unverified gate: "+", ".join(missing)+"; no API request was made."
print({"Jev_status":jev_status,"reason":jev_reason,"calls_completed":len(jev_val)+len(jev_test)})


{'Jev_status': 'skipped_no_free_credit', 'reason': 'Unverified gate: confirmed promotional budget, confirmation that free credits cover frozen sample, zero paid-spend limit, automatic refills disabled, team/account verification reference and time; no API request was made.', 'calls_completed': 0}


## API, preço e termos consultados

Consulta oficial em 2026-09-24: o Swagger/OpenAPI TypeSafe versão 0.2.0 documenta GET /v1/models e POST /v1/systemone. Choice retorna escolha, probabilidades, confiança, modelo e tokens; custo/créditos e latência não constam do schema (latência é medida localmente). O site e anúncio oficial publicam US$ 0,042 por milhão de tokens de entrada e saída gratuita. O acordo Master Customer Agreement, atualizado em 2026-09-19, diz que créditos promocionais são discricionários, saldo é verificado na conta e recargas podem comprar créditos automaticamente. A taxa de créditos varia por conta/modelo. O preço público permite estimar USD, mas não verifica saldo nem segurança de cobrança.

Fontes: [API Swagger](https://api.typesafe.ai/docs), [OpenAPI JSON](https://api.typesafe.ai/openapi.json), [preço oficial](https://typesafe.ai/), [anúncio Jev](https://typesafe.ai/blog/introducing-system-one-models-and-jev), [termos, seção 8.2](https://typesafe.ai/legal/mca).

## Partições e amostra congelada

As partições e group_id vêm do manifesto DBSCAN informado: StratifiedGroupKFold, 5 folds, seed 42; fold 0 teste, fold 1 validação, demais treino. O manifesto registra que o manifesto histórico original de K-means não estava disponível e que as listas foram recuperadas pela lógica salva do notebook; contagens e fingerprint foram conferidos. A estimativa conservadora usa bytes UTF-8 do payload JSON como proxy de até um token por byte, aplicado ao preço público. O provedor não publica a conversão de créditos para USD nem seu tokenizador de cobrança.

Se houver orçamento promocional confirmado, o código escolhe a maior amostra balanceada por classe em validação e teste, com seed 42, antes de qualquer resposta. Sem esse dado, fixa todos os IDs elegíveis e interrompe no gate.

## Métricas, baselines e artefatos

needs_verification conta como abstenção, nunca como acerto. Cobertura é a parcela fake/real; desempenho seletivo considera apenas essas respostas. A regra binária compara, na validação, mapear abstenções para real ou fake pela macro-F1 e congela a vencedora antes de abrir o teste (empate: real). ROC-AUC/AP usam scores disponíveis; Brier apenas para probabilidades. HDBSCAN não entra no ranking binário: não há atribuição documentada para notícias novas.

In [2]:
def native(events):
    if not events: return {}
    y=np.array([int(e["record_id"].startswith("fake/")) for e in events])
    ch=np.array([e["choice"] for e in events],object)
    covered=np.isin(ch,["fake","real"])
    pred=np.array([1 if c=="fake" else 0 if c=="real" else 2 for c in ch])
    tp=int(((y==1)&(pred==1)).sum()); fp=int(((y==0)&(pred==1)).sum()); tn=int(((y==0)&(pred==0)).sum())
    fn_fake=int(((y==1)&(pred!=1)).sum()); fn_true=int(((y==0)&(pred!=0)).sum())
    fp_real=int(((y==1)&(pred==0)).sum())
    p1=tp/(tp+fp) if tp+fp else 0.; r1=tp/(tp+fn_fake) if tp+fn_fake else 0.
    p0=tn/(tn+fp_real) if tn+fp_real else 0.; r0=tn/(tn+fn_true) if tn+fn_true else 0.
    f=lambda p,r:2*p*r/(p+r) if p+r else 0.
    out={"n_samples":len(y),"n_true":int((y==0).sum()),"n_fake":int((y==1).sum()),
        "macro_f1":(f(p0,r0)+f(p1,r1))/2,"balanced_accuracy":(r0+r1)/2,
        "precision_fake":p1,"recall_fake":r1,"f1_fake":f(p1,r1),"fpr_true":fp/int((y==0).sum()),
        "accuracy":float(np.mean((pred==y)&covered)),"coverage":float(covered.mean()),
        "abstention_rate":float((~covered).mean()),
        "selective_accuracy":float(np.mean(pred[covered]==y[covered])) if covered.any() else np.nan,
        "roc_auc":np.nan,"average_precision":np.nan,"brier_score_fake":np.nan}
    p=np.array([e.get("probabilities",{}).get("fake",np.nan) for e in events],float)
    if np.all(np.isfinite(p)) and len(np.unique(p))>1:
        out["roc_auc"]=float(roc_auc_score(y,p)); out["average_precision"]=float(average_precision_score(y,p))
        out["brier_score_fake"]=float(brier_score_loss(y,p))
    return out

latest=latest_events()
jev_test=[latest[r] for r in sample["test"] if r in latest and latest[r].get("status")=="success"]
jev_val=[latest[r] for r in sample["validation"] if r in latest and latest[r].get("status")=="success"]
nm=native(jev_test); bm={}
if jev_test and frozen_mapping:
    y=np.array([int(e["record_id"].startswith("fake/")) for e in jev_test])
    pred=np.array([1 if e["choice"]=="fake" else 0 if e["choice"]=="real" else int(frozen_mapping=="fake") for e in jev_test])
    p=np.array([e.get("probabilities",{}).get("fake",np.nan) for e in jev_test],float)
    bm=evalbin(y,pred,p if np.all(np.isfinite(p)) else None,np.all(np.isfinite(p)))

jev_metrics=[]
for name,vals in (("Jev Choice nativo (3 opções)",nm),("Jev binário (regra de validação)",bm)):
    jev_metrics.append({"method":name,"protocol":"canonical_random_group","partition":"test",
       "status":jev_status,**vals,
       "coverage":nm.get("coverage",np.nan),"abstention_rate":nm.get("abstention_rate",np.nan),
       "selective_accuracy":nm.get("selective_accuracy",np.nan),
       "latency_seconds":float(np.mean([e["latency_seconds"] for e in jev_test])) if jev_test else np.nan,
       "estimated_cost_usd":sum(float(e.get("estimated_cost_usd_from_public_rate",0) or 0) for e in jev_test+jev_val),
       "credits_consumed":"unavailable" if jev_test else 0,"score_type":"Choice probabilities" if jev_test else "unavailable",
       "metric_origin":"TypeSafe API output" if jev_test else "No Jev response; cost gate blocked execution",
       "source_metric_run":None,"binary_abstention_mapping":frozen_mapping})
metrics_out=pd.DataFrame(jev_metrics+basemetrics)
metrics_out.to_csv(RUN_DIR/"metrics.csv",index=False)

predictions=list(basepred)
for rid in sample["test"]:
    e=latest.get(rid)
    if e and e.get("status")=="success":
        p=e.get("probabilities",{})
        predictions.append({"model":"Jev","partition":"test","record_id":rid,"label_for_evaluation_only":int(rid.startswith("fake/")),
         "response":e.get("choice"),"score":p.get("fake"),"score_fake":p.get("fake"),"probabilities_json":json.dumps(p,ensure_ascii=False),
         "confidence":e.get("confidence"),"abstention":e.get("choice")=="needs_verification",
         "latency_seconds":e.get("latency_seconds"),"estimated_cost_usd":e.get("estimated_cost_usd_from_public_rate"),
         "provider_reported_cost_usd":e.get("provider_reported_cost_usd","unavailable"),
         "credits_consumed":e.get("credits_consumed","unavailable"),"usage_input_tokens":e.get("usage_input_tokens"),
         "usage_output_tokens":e.get("usage_output_tokens"),"model_version":e.get("model_version","unavailable"),
         "score_type":"Choice probabilities","state_sent":e.get("request_state")})
    else:
        predictions.append({"model":"Jev","partition":"test","record_id":rid,"label_for_evaluation_only":int(rid.startswith("fake/")),
         "response":None,"score":np.nan,"score_fake":np.nan,"probabilities_json":None,"confidence":np.nan,"abstention":np.nan,
         "latency_seconds":np.nan,"estimated_cost_usd":0.0,"provider_reported_cost_usd":"unavailable_no_call",
         "credits_consumed":0,"usage_input_tokens":np.nan,"usage_output_tokens":np.nan,
         "model_version":"unavailable_no_call","score_type":"unavailable","state_sent":None})
prediction_columns=["model","partition","record_id","label_for_evaluation_only","response","score","score_fake","probabilities_json","confidence","abstention","latency_seconds","estimated_cost_usd","provider_reported_cost_usd","credits_consumed","usage_input_tokens","usage_output_tokens","model_version","score_type","state_sent"]
pd.DataFrame(predictions,columns=prediction_columns).to_csv(RUN_DIR/"predictions.csv",index=False)

sample_info={s:{"n_records":len(sample[s]),"n_true":sum(r.startswith("true/") for r in sample[s]),
 "n_fake":sum(r.startswith("fake/") for r in sample[s]),"record_ids":sample[s],"group_ids":[group[r] for r in sample[s]]} for s in sample}
part_info={s:{"n_records":len(parts[s]),"n_true":sum(r.startswith("true/") for r in parts[s]),
 "n_fake":sum(r.startswith("fake/") for r in parts[s]),"record_ids":parts[s],"group_ids":[group[r] for r in parts[s]]} for s in parts}
run_manifest={"run_id":RUN_ID,"status":jev_status,"reason":jev_reason,
 "created_utc":RUN_STARTED_UTC,"finished_utc":datetime.now(timezone.utc).isoformat(),
 "notebook":"machine-learning/unsupervised-learning/history/deteccao-de-anomalias/jev-standalone.ipynb",
 "environment":{"python":sys.version.split()[0],"pandas":pd.__version__,"numpy":np.__version__,"scikit_learn":sklearn.__version__},
 "corpus":{"name":manifest_src["corpus"]["name"],"revision":manifest_src["corpus"]["revision"],
  "url":manifest_src["corpus"]["url"],"archive_sha256":corpus_sha,"labels":{"0":"True","1":"Fake"}},
 "partition_provenance":{"source_manifest":"machine-learning/outputs/model-comparison/dbscan-20260924T141332Z/run_manifest.json",
  "source_manifest_sha256":hashlib.sha256(SRC_MANIFEST.read_bytes()).hexdigest(),
  "canonical_manifest_available":manifest_src["partition_provenance"]["canonical_requested_manifest_available"],
  "canonical_manifest_blocker":manifest_src["partition_provenance"]["canonical_manifest_blocker"],
  "recovery":manifest_src["partition_provenance"]["recovery"],"split":manifest_src["split"]["canonical"]},
 "partitions":part_info,"sample":{"seed":SEED,"selection":sample_rule,"frozen_before_first_response":True,
  "partial_comparison":len(sample["test"])<len(parts["test"]),"validation_test_group_overlap":False,
  "partitions":sample_info,"max_calls":len(chosen),"call_limit":CALL_CAP},
 "api":{"provider":"TypeSafe AI","openapi_version":API_VERSION,"endpoint":API_BASE+"/v1/systemone",
  "models_endpoint":API_BASE+"/v1/models","model_alias_requested":MODEL_ALIAS,"model_version_resolved":model_resolved,
  "question_name":qname,"question_type":"Choice","choices":options,"question":question,
  "state_fields_sent":["texto_trunc"],"max_text_words":200,
  "credential_source":"TYPESAFE_API_KEY environment only; value never serialized"},
 "cost_gate":{"status":jev_status,"team_confirmed_promotional_budget_usd":TEAM_CONFIRMED_PROMOTIONAL_BUDGET_USD,
  "team_confirmed_credits_cover_sample":TEAM_CONFIRMS_BUDGET_COVERS_FROZEN_SAMPLE,
  "paid_spend_limit_zero_confirmed":TEAM_CONFIRMS_PAID_SPEND_LIMIT_IS_ZERO,
  "automatic_refills_disabled_confirmed":TEAM_CONFIRMS_AUTOMATIC_REFILLS_DISABLED,
  "confirmation_reference":TEAM_CONFIRMATION_REFERENCE,"account_settings_checked_at_utc":ACCOUNT_SETTINGS_CHECKED_AT_UTC,
  "price_terms_sources":PRICE_SOURCES,"public_input_price_usd_per_million_tokens":INPUT_PRICE,
  "public_output_price_usd_per_million_tokens":OUTPUT_PRICE,"input_token_upper_bound":token_bound,
  "estimated_max_cost_usd":round(estimate,8),
  "estimate_method":"UTF-8 JSON payload byte length as <=1 token/byte proxy; provider tokenizer and credit/USD conversion unavailable",
  "realized_api_cost_usd":0.0 if not (jev_test or jev_val) else "unavailable_not_returned_by_api",
  "realized_usage_based_estimate_usd":0.0 if not (jev_test or jev_val) else sum(float(e.get("estimated_cost_usd_from_public_rate",0) or 0) for e in jev_test+jev_val),
  "realized_api_credits":0 if not (jev_test or jev_val) else "unavailable_from_response_schema"},
 "binary_mapping":{"rule":"Validation macro-F1 selects mapping needs_verification to real or fake; tie maps to real; freeze before test.",
  "frozen_mapping":frozen_mapping,"test_used_for_rule_selection":False},
 "baselines":{"source_run":"dbscan-20260924T141332Z","same_test_ids":True,
  "methods":[x[1] for x in specs],"metrics_recalculated_on_selected_test_ids":True,
  "hdbscan_omitted_reason":"No documented score or out-of-sample assignment for unseen news.",
  "historical_incompatible_splits_mixed":False},
 "artifacts":["metrics.csv","predictions.csv","run_manifest.json"]}
(RUN_DIR/"run_manifest.json").write_text(json.dumps(run_manifest,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
print("Artifacts:",RUN_DIR/"metrics.csv",RUN_DIR/"predictions.csv",RUN_DIR/"run_manifest.json")
print("Jev:",jev_status,"; calls:",len(jev_test)+len(jev_val),"; max cost USD:",round(estimate,8))


Artifacts: C:\Users\CUL7CA\Desktop\Eldorado\olimpo-fake-news-ai\machine-learning\outputs\model-comparison\jev-standalone-20260924T164252Z\metrics.csv C:\Users\CUL7CA\Desktop\Eldorado\olimpo-fake-news-ai\machine-learning\outputs\model-comparison\jev-standalone-20260924T164252Z\predictions.csv C:\Users\CUL7CA\Desktop\Eldorado\olimpo-fake-news-ai\machine-learning\outputs\model-comparison\jev-standalone-20260924T164252Z\run_manifest.json
Jev: skipped_no_free_credit ; calls: 0 ; max cost USD: 0.25757126


## Tabela final

Comparação de Jev e baselines no mesmo subconjunto canônico de teste. Origem das métricas locais: run DBSCAN, filtrado e recalculado nos mesmos IDs. Jev fica com status skipped_no_free_credit e sem métricas quando o gate impede chamadas. Esta decisão do Jev não é fact-check comprovado.

In [3]:
from IPython.display import display, Markdown
# Calibration is evaluated only for actual probability outputs, without fitting on test.
calibration_rows=[]
for source_method,display_name in specs:
    q=pred_src[(pred_src.protocol=="canonical_random_group")&(pred_src.partition=="test")&(pred_src.method==source_method)&(pred_src.record_id.isin(set(sample["test"])))].copy()
    if len(q) and str(q.score_type.dropna().iloc[0])=="positive_class_probability":
        q["p_fake"]=pd.to_numeric(q.score,errors="coerce")
        q["y_fake"]=q.record_id.map(lambda r:int(r.startswith("fake/")))
        q=q[np.isfinite(q.p_fake)]
        q["bin"]=np.minimum((q.p_fake*10).astype(int),9)
        for b,g in q.groupby("bin"):
            calibration_rows.append({"method":display_name,"partition":"test","bin":int(b),"n":len(g),
                "mean_probability_fake":g.p_fake.mean(),"observed_fake_fraction":g.y_fake.mean(),
                "probability_origin":"returned model probability; no test calibration"})
if jev_test:
    q=pd.DataFrame([{"record_id":e["record_id"],"p_fake":e.get("probabilities",{}).get("fake",np.nan)} for e in jev_test])
    q["y_fake"]=q.record_id.map(lambda r:int(r.startswith("fake/")))
    q=q[np.isfinite(q.p_fake)]; q["bin"]=np.minimum((q.p_fake*10).astype(int),9)
    for b,g in q.groupby("bin"):
        calibration_rows.append({"method":"Jev","partition":"test","bin":int(b),"n":len(g),
            "mean_probability_fake":g.p_fake.mean(),"observed_fake_fraction":g.y_fake.mean(),
            "probability_origin":"returned Jev probability; no test calibration"})
print("Test calibration bins for available probabilities; no calibration was fitted on test:")
if calibration_rows: display(pd.DataFrame(calibration_rows))
else: print("No probability outputs available.")

cols=["method","status","n_samples","macro_f1","balanced_accuracy","precision_fake","recall_fake","fpr_true",
 "accuracy","roc_auc","average_precision","coverage","abstention_rate","selective_accuracy","brier_score_fake",
 "latency_seconds","estimated_cost_usd","metric_origin"]
t=metrics_out.reindex(columns=cols).copy()
for c in ("macro_f1","balanced_accuracy","precision_fake","recall_fake","fpr_true","accuracy","roc_auc",
 "average_precision","coverage","abstention_rate","selective_accuracy","brier_score_fake"):
    t[c]=t[c].map(lambda x:"—" if pd.isna(x) else f"{float(x):.4f}")
for c in ("latency_seconds","estimated_cost_usd"):
    t[c]=t[c].map(lambda x:"—" if pd.isna(x) else f"{float(x):.6f}")
t["n_samples"]=t["n_samples"].map(lambda x:"—" if pd.isna(x) else str(int(x)))
headers=list(t.columns)
lines=["| "+" | ".join(headers)+" |","| "+" | ".join(["---"]*len(headers))+" |"]
for _,row in t.iterrows():
    vals=[str(row[h]).replace("|","\\|").replace("\n"," ") for h in headers]
    lines.append("| "+" | ".join(vals)+" |")
display(Markdown("\n".join(lines)))
realized=run_manifest["cost_gate"]["realized_api_cost_usd"]
realized_text=f"US$ {realized:.6f}" if isinstance(realized,(int,float)) else str(realized)
display(Markdown(f"Jev: **{jev_status}**. Amostra de teste: {len(sample['test'])}; valida??o: {len(sample['validation'])}; parcial: {len(sample['test'])<len(parts['test'])}. Custo m?ximo estimado: US$ {estimate:.6f}; custo API realizado: {realized_text}."))
display(Markdown("HDBSCAN omitido: sem atribuição/score documentado para notícias novas."))

Test calibration bins for available probabilities; no calibration was fitted on test:


,method,partition,bin,n,mean_probability_fake,observed_fake_fraction,probability_origin
0,Logistic Regression,test,0,198,0.067143,0.000000,returned model probability; no test calibration
1,Logistic Regression,test,1,229,0.146367,0.013100,returned model probability; no test calibration
2,Logistic Regression,test,2,146,0.244456,0.041096,returned model probability; no test calibration
3,Logistic Regression,test,3,115,0.345048,0.130435,returned model probability; no test calibration
4,Logistic Regression,test,4,51,0.446077,0.431373,returned model probability; no test calibration
5,Logistic Regression,test,5,57,0.553139,0.684211,returned model probability; no test calibration
6,Logistic Regression,test,6,85,0.657948,0.905882,returned model probability; no test calibration
7,Logistic Regression,test,7,130,0.756039,0.992308,returned model probability; no test calibration
8,Logistic Regression,test,8,218,0.855770,1.000000,returned model probability; no test calibration
9,Logistic Regression,test,9,211,0.937645,1.000000,returned model probability; no test calibration


| method | status | n_samples | macro_f1 | balanced_accuracy | precision_fake | recall_fake | fpr_true | accuracy | roc_auc | average_precision | coverage | abstention_rate | selective_accuracy | brier_score_fake | latency_seconds | estimated_cost_usd | metric_origin |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Jev Choice nativo (3 opções) | skipped_no_free_credit | — | — | — | — | — | — | — | — | — | — | — | — | — | — | 0.000000 | No Jev response; cost gate blocked execution |
| Jev binário (regra de validação) | skipped_no_free_credit | — | — | — | — | — | — | — | — | — | — | — | — | — | — | 0.000000 | No Jev response; cost gate blocked execution |
| Logistic Regression | executed_local_baseline | 1440 | 0.9493 | 0.9493 | 0.9615 | 0.9361 | 0.0375 | 0.9493 | 0.9895 | 0.9906 | 1.0000 | 0.0000 | 0.9493 | 0.0623 | — | — | Recomputed from matching DBSCAN run predictions.csv |
| LinearSVC | executed_local_baseline | 1440 | 0.9667 | 0.9667 | 0.9786 | 0.9542 | 0.0208 | 0.9667 | 0.9941 | 0.9949 | 1.0000 | 0.0000 | 0.9667 | — | — | — | Recomputed from matching DBSCAN run predictions.csv |
| Random Forest | executed_local_baseline | 1440 | 0.9104 | 0.9104 | 0.9191 | 0.9000 | 0.0792 | 0.9104 | 0.9722 | 0.9754 | 1.0000 | 0.0000 | 0.9104 | 0.1025 | — | — | Recomputed from matching DBSCAN run predictions.csv |
| Isolation Forest | executed_local_baseline | 1440 | 0.9729 | 0.9729 | 0.9620 | 0.9847 | 0.0389 | 0.9729 | 0.9786 | 0.9750 | 1.0000 | 0.0000 | 0.9729 | — | — | — | Recomputed from matching DBSCAN run predictions.csv |
| LOF | executed_local_baseline | 1440 | 0.9479 | 0.9479 | 0.9601 | 0.9347 | 0.0389 | 0.9479 | 0.9766 | 0.9659 | 1.0000 | 0.0000 | 0.9479 | — | — | — | Recomputed from matching DBSCAN run predictions.csv |
| One-Class SVM | executed_local_baseline | 1440 | 0.4685 | 0.5563 | 0.8000 | 0.1500 | 0.0375 | 0.5563 | 0.8738 | 0.8266 | 1.0000 | 0.0000 | 0.5563 | — | — | — | Recomputed from matching DBSCAN run predictions.csv |
| PU Learning | executed_local_baseline | 1440 | 0.7183 | 0.7368 | 0.6566 | 0.9931 | 0.5194 | 0.7368 | 0.9433 | 0.9493 | 1.0000 | 0.0000 | 0.7368 | 0.1823 | — | — | Recomputed from matching DBSCAN run predictions.csv |
| K-means novidade | executed_local_baseline | 1440 | 0.4352 | 0.5382 | 0.7619 | 0.1111 | 0.0347 | 0.5382 | 0.8471 | 0.7751 | 1.0000 | 0.0000 | 0.5382 | — | — | — | Recomputed from matching DBSCAN run predictions.csv |
| DBSCAN novidade | executed_local_baseline | 1440 | 0.9667 | 0.9667 | 0.9504 | 0.9847 | 0.0514 | 0.9667 | 0.9598 | 0.9186 | 1.0000 | 0.0000 | 0.9667 | — | — | — | Recomputed from matching DBSCAN run predictions.csv |

Jev: **skipped_no_free_credit**. Amostra de teste: 1440; valida??o: 1440; parcial: False. Custo m?ximo estimado: US$ 0.257571; custo API realizado: US$ 0.000000.

HDBSCAN omitido: sem atribuição/score documentado para notícias novas.